first lets analyse the data 

In [2]:
import os
import glob
import pandas as pd
import numpy as np
from IPython.display import display

# Helper function to format numbers with spaces
def format_num(n):
    if isinstance(n, float):
        return f"{n:,.2f}".replace(",", " ")
    return f"{n:,}".replace(",", " ")

# Path to the raw CSV files
data_dir = '../results/raw/'
csv_files = glob.glob(os.path.join(data_dir, '*.csv'))

total_files = len(csv_files)
total_rows = 0
total_size_bytes = 0
rows_per_file = []
lifespans = []
start_times = []
end_times = []

# --- New Network Metric Trackers ---
global_delay_sum = 0
global_delay_count = 0
global_plr_sum = 0
global_plr_count = 0

print(f"Found {total_files} CSV files. Processing...")

for file in csv_files:
    try:
        # Get file size
        total_size_bytes += os.path.getsize(file)
        
        # Read the csv file
        df = pd.read_csv(file)
        num_rows = len(df)
        rows_per_file.append(num_rows)
        total_rows += num_rows
        
        # Extract Global Networking Metrics
        if 'AvgMsgDelay' in df.columns:
            global_delay_sum += df['AvgMsgDelay'].sum()
            global_delay_count += df['AvgMsgDelay'].count()
        if 'PacketLossRate' in df.columns:
            global_plr_sum += df['PacketLossRate'].sum()
            global_plr_count += df['PacketLossRate'].count()
        
        # Try to identify a time column to calculate lifespan and simulation duration
        time_col = None
        for col in df.columns:
            if col.lower() in ['time', 't', 'timestamp', 'simtime']:
                time_col = col
                break
        
        if time_col and num_rows > 0:
            start_t = df[time_col].iloc[0]
            end_t = df[time_col].iloc[-1]
            lifespans.append(end_t - start_t)
            start_times.append(start_t)
            end_times.append(end_t)
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Calculate file sizes
total_size_gb = total_size_bytes / (1024 ** 3)
avg_size_mb = (total_size_bytes / total_files) / (1024 ** 2) if total_files > 0 else 0

# Calculate Global Network Averages
global_avg_delay = (global_delay_sum / global_delay_count) if global_delay_count > 0 else 0
global_avg_plr = (global_plr_sum / global_plr_count) if global_plr_count > 0 else 0

# --- Collect Results into a DataFrame ---
results_data = {
    "Metric": [
        "Total CSV files (cars)",
        "Total rows across all files",
        "Total dataset size",
        "Average file size"
    ],
    "Value": [
        f"{format_num(total_files)} files",
        f"{format_num(total_rows)} rows",
        f"{format_num(round(total_size_gb, 2))} GB",
        f"{format_num(round(avg_size_mb, 2))} MB"
    ]
}

if rows_per_file:
    results_data["Metric"].extend([
        "Average rows per file",
        "Highest number of rows",
        "Lowest number of rows"
    ])
    results_data["Value"].extend([
        f"{format_num(int(round(np.mean(rows_per_file))))} rows",
        f"{format_num(np.max(rows_per_file))} rows",
        f"{format_num(np.min(rows_per_file))} rows"
    ])
    
if global_delay_count > 0:
    results_data["Metric"].extend([
        "Global Avg Message Delay",
        "Global Packet Loss Rate"
    ])
    results_data["Value"].extend([
        f"{global_avg_delay:.6f} seconds ({global_avg_delay * 1000:.2f} ms)",
        f"{global_avg_plr * 100:.2f} %"
    ])

avg_lifespan_sec = 0
avg_lifespan_steps = 0
if lifespans:
    # The time column in the CSV is already in seconds (e.g., 17002.1)
    avg_lifespan_sec = np.mean(lifespans)
    avg_lifespan_steps = avg_lifespan_sec * 10
    results_data["Metric"].extend([
        "Average lifespan"
    ])
    results_data["Value"].extend([
        f"{format_num(round(avg_lifespan_sec, 2))} seconds ({format_num(int(round(avg_lifespan_steps)))} steps)"
    ])
    
duration_sec = 0
duration_steps = 0
if start_times and end_times:
    sim_start = min(start_times)
    sim_end = max(end_times)
    # The time difference is in seconds
    duration_sec = sim_end - sim_start
    duration_steps = duration_sec * 10
    results_data["Metric"].extend([
        "Simulation start time",
        "Simulation end time",
        "Total simulation duration"
    ])
    results_data["Value"].extend([
        f"{format_num(round(sim_start, 2))} s",
        f"{format_num(round(sim_end, 2))} s",
        f"{format_num(round(duration_sec, 2))} seconds ({format_num(int(round(duration_steps)))} steps)"
    ])

results_df = pd.DataFrame(results_data)

# Display as a pretty HTML dataframe in Jupyter
display(results_df)

# --- Build and Save Markdown Report ---
report_lines = [
    "# Data Analysis Report\n",
    "## General Information",
    f"- **Total CSV files (cars):** {format_num(total_files)} files",
    f"- **Total rows across all files:** {format_num(total_rows)} rows",
    f"- **Total dataset size:** {format_num(round(total_size_gb, 2))} GB",
    f"- **Average file size:** {format_num(round(avg_size_mb, 2))} MB"
]

if rows_per_file:
    report_lines.extend([
        f"- **Average rows per file:** {format_num(int(round(np.mean(rows_per_file))))} rows",
        f"- **Highest number of rows:** {format_num(np.max(rows_per_file))} rows",
        f"- **Lowest number of rows:** {format_num(np.min(rows_per_file))} rows"
    ])
    
if global_delay_count > 0:
    report_lines.extend([
        "\n## Global Network Health",
        f"- **Global Average Message Delay:** {global_avg_delay:.6f} seconds ({global_avg_delay * 1000:.2f} ms)",
        f"- **Global Average Packet Loss Rate:** {global_avg_plr * 100:.2f} %"
    ])

report_lines.extend(["\n## Temporal Information"])

if lifespans:
    report_lines.append(f"- **Average lifespan of a car:** {format_num(round(avg_lifespan_sec, 2))} seconds ({format_num(int(round(avg_lifespan_steps)))} steps)")
    
if start_times and end_times:
    report_lines.append(f"- **Simulation start time:** {format_num(round(sim_start, 2))} s")
    report_lines.append(f"- **Simulation end time:** {format_num(round(sim_end, 2))} s")
    report_lines.append(f"- **Total simulation duration:** {format_num(round(duration_sec, 2))} seconds ({format_num(int(round(duration_steps)))} steps)")
else:
    report_lines.append("- *(Could not find a recognized time column to calculate durations)*")

report_md = "\n".join(report_lines)

report_path = "analysis_report.md"
with open(report_path, "w") as f:
    f.write(report_md)

print(f"\nReport successfully saved to {os.path.abspath(report_path)}")

Found 1239 CSV files. Processing...


,Metric,Value
0,Total CSV files (cars),1 239 files
1,Total rows across all files,35 099 085 rows
2,Total dataset size,13.14 GB
3,Average file size,10.86 MB
4,Average rows per file,28 329 rows
5,Highest number of rows,71 590 rows
6,Lowest number of rows,235 rows
7,Global Avg Message Delay,0.005237 seconds (5.24 ms)
8,Global Packet Loss Rate,32.61 %
9,Average lifespan,2 832.76 seconds (28 328 steps)



Report successfully saved to /home/massi/Documents/omnetpp-5.6.2/samples/TrajectoryCollector/custom-scripts/analysis_report.md


This new cell loads up the `data_car_0_t17002.csv` file specifically and performs some analytics on it. Here is what it does:
1. **Loads & previews the data:** Shows the shape and prints the first 5 rows in a neat HTML format.
2. **Prints Key Stats:** Calculates and displays total tracked time, average/max speed, and max acceleration/deceleration.
3. **Plots 4 graphs:** Using `matplotlib` and `seaborn`, it will plot:
   - The car's **Trajectory (X vs Y position)** colored by the speed it was traveling.
   - Its **Speed profile** over time.
   - Its **Acceleration profile** over time.
   - Its **Heading/Direction** over time.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")

# Load the specific car data
file_path = '../results/raw/data_car_0_t17002.csv'
df_car0 = pd.read_csv(file_path)

print(f"Data for Car 0 loaded successfully. Shape: {df_car0.shape}")
print("------------------------------------------")
print(f"Columns in the dataset: {df_car0.columns.tolist()}")
print("------------------------------------------")

display(df_car0.head())
display(df_car0.tail())

# --- Basic Analytics ---
print("\n--- Analytics for Car 0 ---")
print(f"Total time tracked: {df_car0['Time'].iloc[-1] - df_car0['Time'].iloc[0]:.2f} seconds")
print(f"Average Speed: {df_car0['Speed'].mean():.2f} m/s")
print(f"Max Speed: {df_car0['Speed'].max():.2f} m/s")
print(f"Max Acceleration: {df_car0['Acceleration'].max():.2f} m/s^2")
print(f"Max Deceleration: {df_car0['Acceleration'].min():.2f} m/s^2")

# Create a figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Trajectory and Analytics for Car 0', fontsize=16)

# 1. Trajectory Plot (X vs Y)
sns.scatterplot(x='X', y='Y', hue='Speed', data=df_car0, palette='viridis', ax=axes[0, 0], edgecolor=None, s=20)
axes[0, 0].set_title('Trajectory (X vs Y) Colored by Speed')
axes[0, 0].set_xlabel('X Position')
axes[0, 0].set_ylabel('Y Position')

# 2. Speed over Time
sns.lineplot(x='Time', y='Speed', data=df_car0, ax=axes[0, 1], color='blue')
axes[0, 1].set_title('Speed Profile over Time')
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Speed (m/s)')

# 3. Acceleration over Time
sns.lineplot(x='Time', y='Acceleration', data=df_car0, ax=axes[1, 0], color='red')
axes[1, 0].set_title('Acceleration Profile over Time')
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Acceleration (m/s²)')

# 4. Heading over Time
sns.lineplot(x='Time', y='Heading', data=df_car0, ax=axes[1, 1], color='green')
axes[1, 1].set_title('Heading Direction over Time')
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('Heading (degrees)')

plt.tight_layout()
plt.show()

1. **Missing Data Analysis:** It calculates absolute missing values, percentages, and (if there are any missing values) charts them visually using `missingno`.
2. **Duplicate Row Checks:** It counts exactly how many rows are completely identical. 
3. **Typing & Uniqueness:** Shows a table detailing the pandas inference datatype, exactly how many unique values exist per column, and gives 3 "example" sample values to verify formats (important for things like detecting if 'Time' or 'LaneID' need specific parsing).
4. **Outlier Detection:** Dynamically loops through every numerical column (ignoring categorical/ID bounds) and renders Boxplots for each to map their spread and visually expose any harsh outliers.
5. **Statistical Distribution Summary:** A transposed DataFrame of Pandas' `.describe()` mapping `.min()`, `.max()`, std variation, means, and percentiles per numeric column. 

In [ ]:
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# File path from previous slice
file_path = '../results/raw/data_car_67_t18027.csv'
df_car0 = pd.read_csv(file_path)

print("--- Data Preprocessing Diagnostics ---")

# 1. Missing Values Analysis
missing_count = df_car0.isnull().sum()
total_cells = np.prod(df_car0.shape)
total_missing = missing_count.sum()
percent_missing = (total_missing / total_cells) * 100

print(f"\n1. Missing Values: {total_missing} cells missing ({percent_missing:.2f}%)")
if total_missing > 0:
    missing_df = pd.DataFrame({'Missing Values': missing_count[missing_count > 0]})
    display(missing_df)
    
    # Visualize missing data if there is any
    plt.figure(figsize=(10, 4))
    msno.matrix(df_car0, sparkline=False, figsize=(10, 4), fontsize=10)
    plt.title("Missing Data Matrix", fontsize=14)
    plt.show()

# 2. Duplicate Rows Analysis
duplicates = df_car0.duplicated().sum()
print(f"\n2. Duplicated Rows: {duplicates}")

# 3. Data Types and Unique Values
print("\n3. Data Types and Unique Value Counts")
info_df = pd.DataFrame({
    'Data Type': df_car0.dtypes,
    'Unique Values': df_car0.nunique(),
    'Example Values': [df_car0[col].unique()[:3] for col in df_car0.columns]
})
display(info_df)

# 4. Outlier Analysis (using Boxplots for numerical columns)
print("\n4. Distribution and Outlier Analysis (Numerical Columns)")
numerical_cols = df_car0.select_dtypes(include=['int64', 'float64']).columns
# Filter out standard non-distributable columns if needed
cols_to_plot = [c for c in numerical_cols if c not in ['Time', 'LaneID']] 

num_plots = len(cols_to_plot)
cols_per_row = 4
rows = (num_plots + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(15, 3*rows))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    sns.boxplot(x=df_car0[col], ax=axes[i], color='skyblue')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')

# Hide any empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# Print basic statistics for numerical columns
print("\n--- Basic Statistics Summary ---")
display(df_car0.describe().transpose())